In [1]:
from pathlib import Path
import numpy as np
import librosa
import spectrograms as sg
from spectrograms import (
    StftParams, SpectrogramParams,
    ITDSpectrogramParams, IPDSpectrogramParams,
    ILDSpectrogramParams, ILRSpectrogramParams,
    compute_itd_spectrogram, compute_itd_spectrogram_diff,
    compute_ipd_spectrogram,
    compute_ild_spectrogram,
    compute_ilr_spectrogram, compute_ilr_spectrogram_diff,
)

# Audio files
ROOT = Path("./audio/ambisonic_examples")
ref_path = ROOT / "castanetsRev_dynamic_A0_A360_E30_HOA_REF_rendered.wav"
test_path = ROOT / "castanetsRev_dynamic_A0_A360_E30_FOA_REF_rendered.wav"

ref_audio, sr = librosa.load(str(ref_path), sr=44100, mono=False)
test_audio, sr = librosa.load(str(test_path), sr=44100, mono=False)

ref_audio = ref_audio.astype(np.float64)
test_audio = test_audio.astype(np.float64)

print(f"Reference audio: {ref_audio.shape}, sr={sr}")
print(f"Test audio:      {test_audio.shape}, sr={sr}")

Reference audio: (2, 408897), sr=44100
Test audio:      (2, 408897), sr=44100


In [2]:
# Common STFT parameters matching Binaspect defaults
stft_params = StftParams(n_fft=4096, hop_size=1024, window=sg.WindowType.hanning, centre=True)
spec_params = SpectrogramParams(stft_params, sr)

## ITD — Interaural Time Difference

Captures the time delay (seconds) between when a sound arrives at each ear.
Primary localisation cue for low frequencies (< ~1.5 kHz).

In [3]:
itd_params = ITDSpectrogramParams(spec_params, start_freq=50.0, end_freq=620.0, magphase_power=1)

ref_itd = compute_itd_spectrogram(ref_audio, itd_params)
print(f"ITD spectrogram shape: {ref_itd.shape}")  # (53, n_frames)

itd_time_diff, mean_diff_degrees, mean_diff_itd = compute_itd_spectrogram_diff(
    ref_audio, test_audio, itd_params
)
print(f"Mean difference: {mean_diff_degrees:.2f}° / {mean_diff_itd*1e6:.2f} µs")
print(f"ITD time diff shape: {itd_time_diff.shape}")

ITD spectrogram shape: (53, 400)
Mean difference: 6.30° / -15.34 µs
ITD time diff shape: (400,)


## IPD — Interaural Phase Difference

Phase difference (radians) between left and right channels. Related to ITD but expressed directly as phase. The `wrapped=True` option constrains values to [-π, π].

In [4]:
ipd_params = IPDSpectrogramParams(spec_params, start_freq=50.0, end_freq=620.0, wrapped=True)

ref_ipd = compute_ipd_spectrogram(ref_audio, ipd_params)
print(f"IPD spectrogram shape: {ref_ipd.shape}")  # (53, n_frames)
print(f"Value range: [{np.nanmin(ref_ipd):.4f}, {np.nanmax(ref_ipd):.4f}] radians")

IPD spectrogram shape: (53, 400)
Value range: [-3.1342, 3.1392] radians


## ILD — Interaural Level Difference

Level difference (dB) between left and right channels. Primary high-frequency localisation cue (> ~1.5 kHz). NaN where one or both channels are silent.

In [5]:
ild_params = ILDSpectrogramParams(spec_params, start_freq=1700.0, end_freq=4600.0)

ref_ild = compute_ild_spectrogram(ref_audio, ild_params)
print(f"ILD spectrogram shape: {ref_ild.shape}")  # (269, n_frames)
print(f"Value range: [{np.nanmin(ref_ild):.2f}, {np.nanmax(ref_ild):.2f}] dB")

ILD spectrogram shape: (269, 400)
Value range: [-48.28, 49.15] dB


## ILR — Interaural Level Ratio

Normalised level ratio in [-1, 1]. Alternative to ILD that avoids dB scaling. Positive = left louder, negative = right louder. NaN where both channels are silent.

In [6]:
ilr_params = ILRSpectrogramParams(spec_params, start_freq=1700.0, end_freq=4600.0)

ref_ilr = compute_ilr_spectrogram(ref_audio, ilr_params)
print(f"ILR spectrogram shape: {ref_ilr.shape}")  # (269, n_frames)
print(f"Value range: [{np.nanmin(ref_ilr):.4f}, {np.nanmax(ref_ilr):.4f}]")

ilr_time_diff, mean_ilr_diff = compute_ilr_spectrogram_diff(ref_audio, test_audio, ilr_params)
print(f"Mean ILR difference: {mean_ilr_diff:.6f}")
print(f"ILR time diff shape: {ilr_time_diff.shape}")

ILR spectrogram shape: (269, 400)
Value range: [-0.9961, 0.9965]
Mean ILR difference: 0.170577
ILR time diff shape: (400,)
